In [1]:
import cudf
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist
from cuml.cluster import HDBSCAN
from pyproj import Transformer
import plotly.express as px
from tqdm import tqdm
import re

# ---------------------------------------------------------
# 1. DATA PREPARATION
# ---------------------------------------------------------
def prepare_traffic_data(filepath):
    """Loads CSV, flattens coordinates to UTM, and extracts temporal/text features."""
    print("Loading and preparing data...")
    gpu_dataset = cudf.read_csv(filepath)
    df = gpu_dataset.dropna(subset=['latitude', 'longitude']).to_pandas().copy()

    # Flatten coordinates
    transformer = Transformer.from_crs("EPSG:4326", "EPSG:32643", always_xy=True)
    df['x_meters'], df['y_meters'] = transformer.transform(df['longitude'].values, df['latitude'].values)

    # Temporal features
    df['created_datetime'] = pd.to_datetime(df['created_datetime'], errors='coerce')
    df['hour'] = df['created_datetime'].dt.hour
    df['day_of_week'] = df['created_datetime'].dt.day_name()

    # Clean violation strings
    def clean_violation(val):
        val_str = str(val)
        if '["' in val_str: return val_str.split('["')[1].split('"')[0]
        return val_str
    df['primary_violation'] = df['violation_type'].apply(clean_violation)
    
    return df

# Helper for aggregating junctions
def _get_top_value(series):
    clean_series = series.dropna()
    return clean_series.mode().iloc[0] if not clean_series.empty else "Unknown"

# Helper for extracting localities (unchanged — returns last clean segment)
def _extract_clean_locality(loc_string):
    if pd.isna(loc_string): return "Unknown"
    parts = [p.strip() for p in str(loc_string).split(',')]
    clean_parts = []
    for p in parts:
        p_lower = p.lower()
        if 'karnataka' in p_lower or 'india' in p_lower: continue
        if 'bengaluru' in p_lower or 'bangalore' in p_lower: continue
        if 'bbmp' in p_lower or 'city corporation' in p_lower: continue
        if re.search(r'\b\d{6}\b', p): continue
        if not p: continue
        clean_parts.append(p)
    return clean_parts[-1] if clean_parts else "Unknown"

# NEW: Helper for extracting street name — returns first clean segment instead of last
def _extract_clean_street(loc_string):
    if pd.isna(loc_string): return "Unknown"
    parts = [p.strip() for p in str(loc_string).split(',')]
    clean_parts = []
    for p in parts:
        p_lower = p.lower()
        if 'karnataka' in p_lower or 'india' in p_lower: continue
        if 'bengaluru' in p_lower or 'bangalore' in p_lower: continue
        if 'bbmp' in p_lower or 'city corporation' in p_lower: continue
        if re.search(r'\b\d{6}\b', p): continue
        if not p: continue
        clean_parts.append(p)
    return clean_parts[0] if clean_parts else "Unknown"

# NEW: Street-grouped HDBSCAN clustering
# Runs HDBSCAN per street-name group to prevent spatial chaining across different streets.
# Groups below min_cluster_size are treated as single small hotspots instead of being discarded.
# Points with street_name == "Unknown" are clustered separately as a fallback group.
MIN_CLUSTER_SIZE = 100
MIN_SAMPLES = 100
SMALL_GROUP_THRESHOLD = MIN_CLUSTER_SIZE
MEGA_GROUP_THRESHOLD = 500  # NEW: streets above this skip HDBSCAN, treated as one hotspot

def _run_street_grouped_hdbscan(df_group_b):
    df = df_group_b.copy()
    df['street_name'] = df['location'].apply(_extract_clean_street)

    b_mapping = {}
    group_b_summaries = []
    all_labels = pd.Series(index=df.index, dtype=object)
    global_cluster_counter = 0

    unknown_mask = df['street_name'] == "Unknown"
    df_unknown = df[unknown_mask].copy()
    df_known = df[~unknown_mask].copy()

    street_counts = df_known.groupby('street_name').size()
    large_streets = street_counts[(street_counts >= SMALL_GROUP_THRESHOLD) & (street_counts < MEGA_GROUP_THRESHOLD)].index  # CHANGED
    small_streets = street_counts[street_counts < SMALL_GROUP_THRESHOLD].index
    mega_streets  = street_counts[street_counts >= MEGA_GROUP_THRESHOLD].index  # NEW

    # Batch-process small street groups
    if len(small_streets) > 0:
        df_small = df_known[df_known['street_name'].isin(small_streets)].copy()
        for street_name, street_df in df_small.groupby('street_name'):
            label_key = f"SMALL-{global_cluster_counter}"
            global_cluster_counter += 1
            top_locality = _get_top_value(street_df['location'].apply(_extract_clean_locality))
            hotspot_name = f"{top_locality} - {street_name} (Small Hotspot)"
            b_mapping[label_key] = hotspot_name
            all_labels.loc[street_df.index] = label_key
            group_b_summaries.append({
                'hotspot_id': label_key,
                'label_name': hotspot_name,
                'label_type': 'small_street_hotspot',
                'total_violations': len(street_df),
                'centroid_lat': round(street_df['latitude'].mean(), 6),
                'centroid_long': round(street_df['longitude'].mean(), 6),
                'top_violation': _get_top_value(street_df['primary_violation']),
                'top_vehicle': _get_top_value(street_df['vehicle_type'])
            })
        print(f" -> Batched {len(small_streets)} small street groups as individual small hotspots.")

    # NEW: Batch-process mega street groups (too large for HDBSCAN, treat as one hotspot)
    if len(mega_streets) > 0:
        df_mega = df_known[df_known['street_name'].isin(mega_streets)].copy()
        for street_name, street_df in df_mega.groupby('street_name'):
            label_key = f"MEGA-{global_cluster_counter}"
            global_cluster_counter += 1
            top_locality = _get_top_value(street_df['location'].apply(_extract_clean_locality))
            hotspot_name = f"{top_locality} - {street_name} (High-Volume Corridor)"
            b_mapping[label_key] = hotspot_name
            all_labels.loc[street_df.index] = label_key
            group_b_summaries.append({
                'hotspot_id': label_key,
                'label_name': hotspot_name,
                'label_type': 'unofficial_hotspot',
                'total_violations': len(street_df),
                'centroid_lat': round(street_df['latitude'].mean(), 6),
                'centroid_long': round(street_df['longitude'].mean(), 6),
                'top_violation': _get_top_value(street_df['primary_violation']),
                'top_vehicle': _get_top_value(street_df['vehicle_type'])
            })
        print(f" -> Batched {len(mega_streets)} mega street groups ({MEGA_GROUP_THRESHOLD}+ pts) as high-volume corridors.")

    # HDBSCAN only on large street groups (100–4999 points)
    df_large = df_known[df_known['street_name'].isin(large_streets)].copy()
    print(f" -> Running HDBSCAN on {len(large_streets)} large street groups...")
    for street_name, street_df in tqdm(df_large.groupby('street_name'), desc="Street-grouped HDBSCAN (large streets only)"):
        print(f"   Processing: '{street_name}' | points: {len(street_df):,}")  # visibility into which street
        gpu_coords = cudf.DataFrame(street_df[['x_meters', 'y_meters']])
        model = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES, output_type='numpy')
        raw_labels = model.fit_predict(gpu_coords)

        for raw_id in np.unique(raw_labels):
            cluster_mask = raw_labels == raw_id
            cluster_points = street_df.iloc[cluster_mask]
            if raw_id == -1:
                label_key = f"NOISE-{global_cluster_counter}"
                global_cluster_counter += 1
                hotspot_name = "Transit Noise / Isolated"
            else:
                label_key = f"B-{global_cluster_counter}"
                global_cluster_counter += 1
                top_locality = _get_top_value(cluster_points['location'].apply(_extract_clean_locality))
                hotspot_name = f"{top_locality} (Discovered Area)"
                group_b_summaries.append({
                    'hotspot_id': label_key,
                    'label_name': hotspot_name,
                    'label_type': 'unofficial_hotspot',
                    'total_violations': len(cluster_points),
                    'centroid_lat': round(cluster_points['latitude'].mean(), 6),
                    'centroid_long': round(cluster_points['longitude'].mean(), 6),
                    'top_violation': _get_top_value(cluster_points['primary_violation']),
                    'top_vehicle': _get_top_value(cluster_points['vehicle_type'])
                })
            b_mapping[label_key] = hotspot_name
            all_labels.loc[cluster_points.index] = label_key

    # Fallback: unknown-street rows
    if not df_unknown.empty:
        n_unknown = len(df_unknown)
        if n_unknown < SMALL_GROUP_THRESHOLD:
            label_key = f"SMALL-{global_cluster_counter}"
            global_cluster_counter += 1
            hotspot_name = "Unclear Address (Small Hotspot)"
            b_mapping[label_key] = hotspot_name
            all_labels.loc[df_unknown.index] = label_key
            group_b_summaries.append({
                'hotspot_id': label_key,
                'label_name': hotspot_name,
                'label_type': 'small_street_hotspot',
                'total_violations': n_unknown,
                'centroid_lat': round(df_unknown['latitude'].mean(), 6),
                'centroid_long': round(df_unknown['longitude'].mean(), 6),
                'top_violation': _get_top_value(df_unknown['primary_violation']),
                'top_vehicle': _get_top_value(df_unknown['vehicle_type'])
            })
        else:
            print(f" -> Running fallback HDBSCAN on {n_unknown} unknown-street points...")
            gpu_coords_unk = cudf.DataFrame(df_unknown[['x_meters', 'y_meters']])
            model_unk = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES, output_type='numpy')
            raw_labels_unk = model_unk.fit_predict(gpu_coords_unk)

            for raw_id in np.unique(raw_labels_unk):
                cluster_mask = raw_labels_unk == raw_id
                cluster_points = df_unknown.iloc[cluster_mask]
                if raw_id == -1:
                    label_key = f"NOISE-{global_cluster_counter}"
                    global_cluster_counter += 1
                    hotspot_name = "Transit Noise / Isolated"
                else:
                    label_key = f"B-{global_cluster_counter}"
                    global_cluster_counter += 1
                    top_locality = _get_top_value(cluster_points['location'].apply(_extract_clean_locality))
                    hotspot_name = f"{top_locality} (Discovered Area - Unclear Address)"
                    group_b_summaries.append({
                        'hotspot_id': label_key,
                        'label_name': hotspot_name,
                        'label_type': 'unofficial_hotspot',
                        'total_violations': len(cluster_points),
                        'centroid_lat': round(cluster_points['latitude'].mean(), 6),
                        'centroid_long': round(cluster_points['longitude'].mean(), 6),
                        'top_violation': _get_top_value(cluster_points['primary_violation']),
                        'top_vehicle': _get_top_value(cluster_points['vehicle_type'])
                    })
                b_mapping[label_key] = hotspot_name
                all_labels.loc[cluster_points.index] = label_key

    df['final_cluster_label'] = all_labels
    return df, b_mapping, group_b_summaries


# ---------------------------------------------------------
# 2. PIPELINE A: STANDARD HYBRID
# ---------------------------------------------------------
def run_standard_hybrid_pipeline(df):
    """Runs the basic Hybrid approach (No Footprint Reclamation)."""
    print("\n--- Running Standard Hybrid Pipeline ---")
    group_a = df[df['junction_name'] != 'No Junction'].copy()
    group_b = df[df['junction_name'] == 'No Junction'].copy()

    # Aggregate Group A
    group_a_summaries = []
    for junction_name, group in tqdm(group_a.groupby('junction_name'), desc="Aggregating Known Junctions"):
        group_a_summaries.append({
            'hotspot_id': f"A-{len(group_a_summaries)+1}",
            'label_name': junction_name,
            'label_type': 'confirmed_junction',
            'total_violations': len(group),
            'centroid_lat': round(group['latitude'].mean(), 6),
            'centroid_long': round(group['longitude'].mean(), 6),
            'top_violation': _get_top_value(group['primary_violation']),
            'top_vehicle': _get_top_value(group['vehicle_type'])
        })
    
    # Granular data for Group A
    viz_a = group_a[['latitude', 'longitude', 'junction_name']].copy()
    viz_a.rename(columns={'junction_name': 'Hotspot_Name'}, inplace=True)

    # NEW: Street-grouped HDBSCAN on Group B (replaces single HDBSCAN call)
    group_b_clustered, b_mapping, group_b_summaries = _run_street_grouped_hdbscan(group_b)

    # Granular data for Group B — map final_cluster_label to human-readable name
    viz_b = group_b_clustered[['latitude', 'longitude', 'final_cluster_label']].copy()
    viz_b['Hotspot_Name'] = viz_b['final_cluster_label'].map(b_mapping)
    viz_b.drop(columns=['final_cluster_label'], inplace=True)

    # Combine everything
    summary_df = pd.concat([pd.DataFrame(group_a_summaries), pd.DataFrame(group_b_summaries)], ignore_index=True)
    summary_df = summary_df.sort_values(by='total_violations', ascending=False).reset_index(drop=True)
    granular_df = pd.concat([viz_a, viz_b], ignore_index=True)

    return summary_df, granular_df


# ---------------------------------------------------------
# 3. PIPELINE B: FOOTPRINT RECLAMATION
# ---------------------------------------------------------
def run_footprint_pipeline(df):
    """Runs the advanced Footprint approach (Reclaiming mislabeled points)."""
    print("\n--- Running Footprint Reclamation Pipeline ---")
    group_a = df[df['junction_name'] != 'No Junction'].copy()
    group_b = df[df['junction_name'] == 'No Junction'].copy()

    # Footprint Calculation
    centroids = group_a.groupby('junction_name')[['x_meters', 'y_meters']].mean().reset_index()
    centroids.rename(columns={'x_meters': 'cx', 'y_meters': 'cy'}, inplace=True)
    
    group_a = group_a.merge(centroids, on='junction_name')
    group_a['dist'] = np.sqrt((group_a['x_meters'] - group_a['cx'])**2 + (group_a['y_meters'] - group_a['cy'])**2)
    
    radii = group_a.groupby('junction_name')['dist'].quantile(0.95).reset_index(name='radius_95')
    j_lookup = centroids.merge(radii, on='junction_name')

    # Distance Matrix & Reclamation
    dist_matrix = cdist(group_b[['x_meters', 'y_meters']].values, j_lookup[['cx', 'cy']].values, metric='euclidean')
    valid_dists = np.where(dist_matrix <= j_lookup['radius_95'].values, dist_matrix, np.inf)
    
    min_dist = np.min(valid_dists, axis=1)
    best_junc_idx = np.argmin(valid_dists, axis=1)
    reclaimed_mask = min_dist != np.inf
    
    print(f" -> Reclaimed {reclaimed_mask.sum()} mislabeled points from Group B!")

    # Assign reclaimed points
    group_b['new_junction'] = 'No Junction'
    group_b.loc[reclaimed_mask, 'new_junction'] = j_lookup['junction_name'].iloc[best_junc_idx[reclaimed_mask]].values
    
    reclaimed_df = group_b[group_b['new_junction'] != 'No Junction'].copy()
    reclaimed_df['junction_name'] = reclaimed_df['new_junction']
    true_group_b = group_b[group_b['new_junction'] == 'No Junction'].copy()

    final_group_a = pd.concat([group_a, reclaimed_df], ignore_index=True)

    # Aggregate Final Group A
    group_a_summaries = []
    for junction_name, group in tqdm(final_group_a.groupby('junction_name'), desc="Aggregating Enriched Group A"):
        group_a_summaries.append({
            'hotspot_id': f"A-{len(group_a_summaries)+1}",
            'label_name': junction_name,
            'label_type': 'confirmed_junction',
            'total_violations': len(group),
            'centroid_lat': round(group['latitude'].mean(), 6),
            'centroid_long': round(group['longitude'].mean(), 6),
            'top_violation': _get_top_value(group['primary_violation']),
            'top_vehicle': _get_top_value(group['vehicle_type'])
        })

    viz_a = final_group_a[['latitude', 'longitude', 'junction_name']].copy()
    viz_a.rename(columns={'junction_name': 'Hotspot_Name'}, inplace=True)

    # NEW: Street-grouped HDBSCAN on true_group_b (replaces single HDBSCAN call)
    true_group_b_clustered, b_mapping, group_b_summaries = _run_street_grouped_hdbscan(true_group_b)

    # Granular data for Group B — map final_cluster_label to human-readable name
    viz_b = true_group_b_clustered[['latitude', 'longitude', 'final_cluster_label']].copy()
    viz_b['Hotspot_Name'] = viz_b['final_cluster_label'].map(b_mapping)
    viz_b.drop(columns=['final_cluster_label'], inplace=True)

    # Combine everything
    summary_df = pd.concat([pd.DataFrame(group_a_summaries), pd.DataFrame(group_b_summaries)], ignore_index=True)
    summary_df = summary_df.sort_values(by='total_violations', ascending=False).reset_index(drop=True)
    granular_df = pd.concat([viz_a, viz_b], ignore_index=True)

    return summary_df, granular_df


# ---------------------------------------------------------
# 4. VISUALIZATION FUNCTION
# ---------------------------------------------------------
def save_granular_map_html(granular_df, title_text, filename):
    """Saves the high-density interactive map to an HTML file."""
    print(f"Generating granular interactive map for {filename}...")
    
    fig = px.scatter_mapbox(
        granular_df,
        lat="latitude",
        lon="longitude",
        color="Hotspot_Name", 
        hover_name="Hotspot_Name",
        zoom=10,
        title=title_text
    )

    fig.update_traces(marker=dict(size=3, opacity=0.6))

    fig.update_layout(
        mapbox_style="carto-darkmatter", 
        margin={"r":0,"t":40,"l":0,"b":0},
        showlegend=False
    )

    save_path = f"{filename}.html"
    fig.write_html(save_path)
    print(f" -> Successfully saved interactive map to: {save_path}")


# ---------------------------------------------------------
# 5. SUMMARY MAP VISUALIZATION
# ---------------------------------------------------------
def save_summary_map_html(summary_df, title_text, filename):
    """Saves a hotspot-level summary map (one marker per hotspot, sized by violation count)."""
    print(f"Generating summary map for {filename}...")

    df = summary_df.dropna(subset=['centroid_lat', 'centroid_long']).copy()

    # Marker size: scale total_violations to a reasonable pixel range
    max_v = df['total_violations'].max()
    min_v = df['total_violations'].min()
    df['marker_size'] = 6 + 30 * ((df['total_violations'] - min_v) / (max_v - min_v + 1))

    # Distinguish label_type via opacity: confirmed junctions fully opaque, others stepped down
    opacity_map = {
        'confirmed_junction':   1.0,
        'unofficial_hotspot':   0.75,
        'small_street_hotspot': 0.5,
    }
    df['marker_opacity'] = df['label_type'].map(opacity_map).fillna(0.6)

    # Custom hover text
    df['hover_text'] = (
        '<b>' + df['label_name'].astype(str) + '</b><br>' +
        'Violations: '    + df['total_violations'].astype(str) + '<br>' +
        'Top violation: ' + df['top_violation'].astype(str)    + '<br>' +
        'Top vehicle: '   + df['top_vehicle'].astype(str)      + '<br>' +
        'Type: '          + df['label_type'].astype(str)
    )

    # Plot each label_type as a separate trace so they appear as distinct legend entries
    label_type_styles = {
        'confirmed_junction':   ('circle',    1.0),
        'unofficial_hotspot':   ('circle',    0.75),
        'small_street_hotspot': ('circle',    0.5),
    }

    import plotly.graph_objects as go

    fig = go.Figure()

    for label_type, (symbol, opacity) in label_type_styles.items():
        subset = df[df['label_type'] == label_type]
        if subset.empty:
            continue
        fig.add_trace(go.Scattermapbox(
            lat=subset['centroid_lat'],
            lon=subset['centroid_long'],
            mode='markers',
            name=label_type.replace('_', ' ').title(),
            text=subset['hover_text'],
            hovertemplate='%{text}<extra></extra>',
            marker=dict(
                size=subset['marker_size'],
                color=subset['total_violations'],
                colorscale='YlOrRd',
                opacity=opacity,
                sizemode='diameter',
                showscale=(label_type == 'confirmed_junction'),  # show colorbar once
                colorbar=dict(title='Violations') if label_type == 'confirmed_junction' else None,
            ),
        ))

    fig.update_layout(
        mapbox=dict(style='carto-darkmatter', zoom=10,
                    center=dict(lat=df['centroid_lat'].mean(), lon=df['centroid_long'].mean())),
        margin={'r': 0, 't': 40, 'l': 0, 'b': 0},
        title=title_text,
        legend=dict(
            title='Hotspot Type',
            bgcolor='rgba(30,30,30,0.7)',
            font=dict(color='white'),
        ),
    )

    save_path = f"{filename}.html"
    fig.write_html(save_path)
    print(f" -> Successfully saved summary map to: {save_path}")

# ---------------------------------------------------------
# 6. DIAGNOSTIC / AUDIT REPORT
# ---------------------------------------------------------
def print_audit_report(
    pipeline_name,
    input_df,
    summary_df,
    granular_df,
    group_a_size,
    group_b_size_original,
    group_b_size_after_reclaim=None,   # None for Pipeline A
    reclaimed_count=None,              # None for Pipeline A
    large_street_count=None,
    small_street_count=None,
):
    """Prints a reconciliation audit report for a completed pipeline run."""

    sep  = "=" * 62
    sep2 = "-" * 62

    print(f"\n{sep}")
    print(f"  AUDIT REPORT — {pipeline_name}")
    print(sep)

    # --- 1. Input size ---
    total_input = len(input_df)
    print(f"\n  [1] INPUT")
    print(f"      Total input rows            : {total_input:>10,}")

    # --- 2. Group A / B split ---
    print(f"\n  [2] INITIAL GROUP SPLIT")
    print(f"      Group A (known junctions)   : {group_a_size:>10,}")
    print(f"      Group B (No Junction)       : {group_b_size_original:>10,}")
    ab_sum = group_a_size + group_b_size_original
    match  = "OK" if ab_sum == total_input else f"MISMATCH (sum={ab_sum:,})"
    print(f"      A + B sum                   : {ab_sum:>10,}  [{match}]")

    # --- 3. Footprint reclamation (Pipeline B only) ---
    if reclaimed_count is not None:
        print(f"\n  [3] FOOTPRINT RECLAMATION")
        print(f"      Points reclaimed B -> A     : {reclaimed_count:>10,}")
        print(f"      Group A after reclaim       : {group_a_size + reclaimed_count:>10,}")
        print(f"      Group B after reclaim       : {group_b_size_after_reclaim:>10,}")
        post_sum = (group_a_size + reclaimed_count) + group_b_size_after_reclaim
        post_match = "OK" if post_sum == total_input else f"MISMATCH (sum={post_sum:,})"
        print(f"      Post-reclaim sum            : {post_sum:>10,}  [{post_match}]")
    else:
        print(f"\n  [3] FOOTPRINT RECLAMATION      :   N/A (Pipeline A)")

    # --- 4. summary_df breakdown by label_type ---
    print(f"\n  [4] SUMMARY_DF BREAKDOWN BY LABEL TYPE")
    type_order = ['confirmed_junction', 'unofficial_hotspot', 'small_street_hotspot']
    all_types  = summary_df['label_type'].unique().tolist()
    for lt in type_order + [t for t in all_types if t not in type_order]:
        subset = summary_df[summary_df['label_type'] == lt]
        n_groups = len(subset)
        n_viols  = subset['total_violations'].sum()
        print(f"      {lt:<30}: {n_groups:>5} groups  |  {n_viols:>9,} violations")

    # --- 5. Grand total reconciliation ---
    print(f"\n  [5] VIOLATION COUNT RECONCILIATION")
    grand_total_in_summary = summary_df['total_violations'].sum()
    gap        = total_input - grand_total_in_summary
    gap_pct    = (gap / total_input * 100) if total_input > 0 else 0
    print(f"      Total input rows            : {total_input:>10,}")
    print(f"      Total violations in summary : {grand_total_in_summary:>10,}")
    print(f"      Gap (noise / unaccounted)   : {gap:>10,}  ({gap_pct:.2f}% of input)")

    # --- 6. Cluster / group counts ---
    n_confirmed  = len(summary_df[summary_df['label_type'] == 'confirmed_junction'])
    n_unofficial = len(summary_df[summary_df['label_type'] == 'unofficial_hotspot'])
    n_small      = len(summary_df[summary_df['label_type'] == 'small_street_hotspot'])

    # Noise points: rows in granular_df whose Hotspot_Name is the noise label
    noise_rows = granular_df[granular_df['Hotspot_Name'] == 'Transit Noise / Isolated']
    n_noise    = len(noise_rows)

    print(f"\n  [6] CLUSTER / GROUP COUNTS")
    print(f"      Confirmed junctions         : {n_confirmed:>10,}")
    print(f"      Unofficial hotspots         : {n_unofficial:>10,}")
    print(f"      Small street hotspots       : {n_small:>10,}")
    if large_street_count is not None:
        print(f"      Large street groups (HDBSCAN): {large_street_count:>9,}")
    if small_street_count is not None:
        print(f"      Small street groups (batched): {small_street_count:>9,}")
    print(f"      Noise points (Transit/Iso.) : {n_noise:>10,}  (excluded from summary)")

    # --- 7. Top 10 hotspots ---
    print(f"\n  [7] TOP 10 HOTSPOTS BY VIOLATIONS")
    print(f"  {sep2}")
    top10 = summary_df.nlargest(10, 'total_violations')[
        ['label_name', 'label_type', 'total_violations']
    ].reset_index(drop=True)
    for i, row in top10.iterrows():
        rank      = f"#{i+1:<3}"
        name      = row['label_name'][:38].ljust(38)
        ltype     = f"[{row['label_type'][:20]}]"
        viols     = f"{row['total_violations']:>8,}"
        print(f"  {rank}  {name}  {ltype:<22}  {viols}")
    print(f"  {sep2}")
    print(f"\n{sep}\n")

<frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.cuda module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.driver module instead.


In [ ]:
file_path = "jan to may police violation_anonymized791b166.csv"
base_traffic_df = prepare_traffic_data(file_path)

# --- Pipeline A ---
summary_standard, granular_standard = run_standard_hybrid_pipeline(base_traffic_df)
save_granular_map_html(granular_standard, "Bengaluru Hotspots (Standard Hybrid)", "approach_A")
save_summary_map_html(summary_standard, "Bengaluru Hotspot Summary (Standard Hybrid)", "approach_A_summary")
print_audit_report(
    pipeline_name         = "STANDARD HYBRID (Pipeline A)",
    input_df              = base_traffic_df,
    summary_df            = summary_standard,
    granular_df           = granular_standard,
    group_a_size          = len(base_traffic_df[base_traffic_df['junction_name'] != 'No Junction']),
    group_b_size_original = len(base_traffic_df[base_traffic_df['junction_name'] == 'No Junction']),
)

# --- Pipeline B ---
# summary_footprint, granular_footprint = run_footprint_pipeline(base_traffic_df)
# save_granular_map_html(granular_footprint, "Bengaluru Hotspots (Footprint Reclamation)", "approach_B")
# save_summary_map_html(summary_footprint, "Bengaluru Hotspot Summary (Footprint Reclamation)", "approach_B_summary")
# print_audit_report(
#     pipeline_name              = "FOOTPRINT RECLAMATION (Pipeline B)",
#     input_df                   = base_traffic_df,
#     summary_df                 = summary_footprint,
#     granular_df                = granular_footprint,
#     group_a_size               = len(base_traffic_df[base_traffic_df['junction_name'] != 'No Junction']),
#     group_b_size_original      = len(base_traffic_df[base_traffic_df['junction_name'] == 'No Junction']),
#     group_b_size_after_reclaim = len(base_traffic_df[base_traffic_df['junction_name'] == 'No Junction']) - 11788,
#     reclaimed_count            = 11788,
# )

In [3]:
# violations_dataset = pd.read_csv("jan to may police violation_anonymized791b166.csv")

In [ ]:
# =============================================================
# PREREQUISITE: Enrich granular_df with raw columns
# Modify run_standard_hybrid_pipeline and run_footprint_pipeline
# to retain extra columns in granular_df
# =============================================================

EXTRA_COLS = [
    'primary_violation', 'vehicle_type', 'hour',
    'created_datetime', 'vehicle_number', 'created_by_id',
    'validation_timestamp', 'police_station', 'center_code'
]

# Patch _run_street_grouped_hdbscan to carry extra cols through viz_b
def _run_street_grouped_hdbscan_enriched(df_group_b):
    """Same as _run_street_grouped_hdbscan but retains EXTRA_COLS in returned df."""
    df = df_group_b.copy()
    df['street_name'] = df['location'].apply(_extract_clean_street)

    b_mapping = {}
    group_b_summaries = []
    all_labels = pd.Series(index=df.index, dtype=object)
    global_cluster_counter = 0

    unknown_mask = df['street_name'] == "Unknown"
    df_unknown = df[unknown_mask].copy()
    df_known   = df[~unknown_mask].copy()

    street_counts = df_known.groupby('street_name').size()
    large_streets = street_counts[(street_counts >= SMALL_GROUP_THRESHOLD) & (street_counts < MEGA_GROUP_THRESHOLD)].index
    small_streets = street_counts[street_counts < SMALL_GROUP_THRESHOLD].index
    mega_streets  = street_counts[street_counts >= MEGA_GROUP_THRESHOLD].index

    # --- Small groups ---
    if len(small_streets) > 0:
        df_small = df_known[df_known['street_name'].isin(small_streets)].copy()
        for street_name, street_df in df_small.groupby('street_name'):
            label_key    = f"SMALL-{global_cluster_counter}"; global_cluster_counter += 1
            top_locality = _get_top_value(street_df['location'].apply(_extract_clean_locality))
            hotspot_name = f"{top_locality} - {street_name} (Small Hotspot)"
            b_mapping[label_key] = hotspot_name
            all_labels.loc[street_df.index] = label_key
            group_b_summaries.append({
                'hotspot_id': label_key, 'label_name': hotspot_name,
                'label_type': 'small_street_hotspot', 'total_violations': len(street_df),
                'centroid_lat': round(street_df['latitude'].mean(), 6),
                'centroid_long': round(street_df['longitude'].mean(), 6),
                'top_violation': _get_top_value(street_df['primary_violation']),
                'top_vehicle':   _get_top_value(street_df['vehicle_type'])
            })
        print(f" -> Batched {len(small_streets)} small street groups.")

    # --- Mega groups ---
    if len(mega_streets) > 0:
        df_mega = df_known[df_known['street_name'].isin(mega_streets)].copy()
        for street_name, street_df in df_mega.groupby('street_name'):
            label_key    = f"MEGA-{global_cluster_counter}"; global_cluster_counter += 1
            top_locality = _get_top_value(street_df['location'].apply(_extract_clean_locality))
            hotspot_name = f"{top_locality} - {street_name} (High-Volume Corridor)"
            b_mapping[label_key] = hotspot_name
            all_labels.loc[street_df.index] = label_key
            group_b_summaries.append({
                'hotspot_id': label_key, 'label_name': hotspot_name,
                'label_type': 'unofficial_hotspot', 'total_violations': len(street_df),
                'centroid_lat': round(street_df['latitude'].mean(), 6),
                'centroid_long': round(street_df['longitude'].mean(), 6),
                'top_violation': _get_top_value(street_df['primary_violation']),
                'top_vehicle':   _get_top_value(street_df['vehicle_type'])
            })
        print(f" -> Batched {len(mega_streets)} mega street groups ({MEGA_GROUP_THRESHOLD}+ pts).")

    # --- Large groups: HDBSCAN ---
    df_large = df_known[df_known['street_name'].isin(large_streets)].copy()
    print(f" -> Running HDBSCAN on {len(large_streets)} large street groups...")
    for street_name, street_df in tqdm(df_large.groupby('street_name'), desc="Street-grouped HDBSCAN"):
        print(f"   Processing: '{street_name}' | points: {len(street_df):,}")
        gpu_coords = cudf.DataFrame(street_df[['x_meters', 'y_meters']])
        model      = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES, output_type='numpy')
        raw_labels = model.fit_predict(gpu_coords)
        for raw_id in np.unique(raw_labels):
            cluster_mask   = raw_labels == raw_id
            cluster_points = street_df.iloc[cluster_mask]
            if raw_id == -1:
                label_key    = f"NOISE-{global_cluster_counter}"; global_cluster_counter += 1
                hotspot_name = "Transit Noise / Isolated"
            else:
                label_key    = f"B-{global_cluster_counter}"; global_cluster_counter += 1
                top_locality = _get_top_value(cluster_points['location'].apply(_extract_clean_locality))
                hotspot_name = f"{top_locality} (Discovered Area)"
                group_b_summaries.append({
                    'hotspot_id': label_key, 'label_name': hotspot_name,
                    'label_type': 'unofficial_hotspot', 'total_violations': len(cluster_points),
                    'centroid_lat': round(cluster_points['latitude'].mean(), 6),
                    'centroid_long': round(cluster_points['longitude'].mean(), 6),
                    'top_violation': _get_top_value(cluster_points['primary_violation']),
                    'top_vehicle':   _get_top_value(cluster_points['vehicle_type'])
                })
            b_mapping[label_key] = hotspot_name
            all_labels.loc[cluster_points.index] = label_key

    # --- Fallback: unknown-street rows ---
    if not df_unknown.empty:
        n_unknown = len(df_unknown)
        if n_unknown < SMALL_GROUP_THRESHOLD:
            label_key    = f"SMALL-{global_cluster_counter}"; global_cluster_counter += 1
            hotspot_name = "Unclear Address (Small Hotspot)"
            b_mapping[label_key] = hotspot_name
            all_labels.loc[df_unknown.index] = label_key
            group_b_summaries.append({
                'hotspot_id': label_key, 'label_name': hotspot_name,
                'label_type': 'small_street_hotspot', 'total_violations': n_unknown,
                'centroid_lat': round(df_unknown['latitude'].mean(), 6),
                'centroid_long': round(df_unknown['longitude'].mean(), 6),
                'top_violation': _get_top_value(df_unknown['primary_violation']),
                'top_vehicle':   _get_top_value(df_unknown['vehicle_type'])
            })
        else:
            print(f" -> Running fallback HDBSCAN on {n_unknown} unknown-street points...")
            gpu_coords_unk  = cudf.DataFrame(df_unknown[['x_meters', 'y_meters']])
            model_unk       = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES, output_type='numpy')
            raw_labels_unk  = model_unk.fit_predict(gpu_coords_unk)
            for raw_id in np.unique(raw_labels_unk):
                cluster_mask   = raw_labels_unk == raw_id
                cluster_points = df_unknown.iloc[cluster_mask]
                if raw_id == -1:
                    label_key    = f"NOISE-{global_cluster_counter}"; global_cluster_counter += 1
                    hotspot_name = "Transit Noise / Isolated"
                else:
                    label_key    = f"B-{global_cluster_counter}"; global_cluster_counter += 1
                    top_locality = _get_top_value(cluster_points['location'].apply(_extract_clean_locality))
                    hotspot_name = f"{top_locality} (Discovered Area - Unclear Address)"
                    group_b_summaries.append({
                        'hotspot_id': label_key, 'label_name': hotspot_name,
                        'label_type': 'unofficial_hotspot', 'total_violations': len(cluster_points),
                        'centroid_lat': round(cluster_points['latitude'].mean(), 6),
                        'centroid_long': round(cluster_points['longitude'].mean(), 6),
                        'top_violation': _get_top_value(cluster_points['primary_violation']),
                        'top_vehicle':   _get_top_value(cluster_points['vehicle_type'])
                    })
                b_mapping[label_key] = hotspot_name
                all_labels.loc[cluster_points.index] = label_key

    df['final_cluster_label'] = all_labels
    return df, b_mapping, group_b_summaries


def run_standard_hybrid_pipeline_enriched(df):
    """Pipeline A — granular_df retains EXTRA_COLS."""
    print("\n--- Running Standard Hybrid Pipeline (Enriched) ---")
    group_a = df[df['junction_name'] != 'No Junction'].copy()
    group_b = df[df['junction_name'] == 'No Junction'].copy()

    group_a_summaries = []
    for junction_name, group in tqdm(group_a.groupby('junction_name'), desc="Aggregating Known Junctions"):
        group_a_summaries.append({
            'hotspot_id': f"A-{len(group_a_summaries)+1}", 'label_name': junction_name,
            'label_type': 'confirmed_junction', 'total_violations': len(group),
            'centroid_lat': round(group['latitude'].mean(), 6),
            'centroid_long': round(group['longitude'].mean(), 6),
            'top_violation': _get_top_value(group['primary_violation']),
            'top_vehicle':   _get_top_value(group['vehicle_type'])
        })

    # viz_a: keep extra cols, add Hotspot_Name
    keep_a      = ['latitude', 'longitude', 'junction_name'] + [c for c in EXTRA_COLS if c in group_a.columns]
    viz_a       = group_a[keep_a].copy()
    viz_a.rename(columns={'junction_name': 'Hotspot_Name'}, inplace=True)

    group_b_clustered, b_mapping, group_b_summaries = _run_street_grouped_hdbscan_enriched(group_b)

    # viz_b: keep extra cols, map label to name
    keep_b      = ['latitude', 'longitude', 'final_cluster_label'] + [c for c in EXTRA_COLS if c in group_b_clustered.columns]
    viz_b       = group_b_clustered[keep_b].copy()
    viz_b['Hotspot_Name'] = viz_b['final_cluster_label'].map(b_mapping)
    viz_b.drop(columns=['final_cluster_label'], inplace=True)

    summary_df  = pd.concat([pd.DataFrame(group_a_summaries), pd.DataFrame(group_b_summaries)], ignore_index=True)
    summary_df  = summary_df.sort_values(by='total_violations', ascending=False).reset_index(drop=True)
    granular_df = pd.concat([viz_a, viz_b], ignore_index=True)

    return summary_df, granular_df


def run_footprint_pipeline_enriched(df):
    """Pipeline B — granular_df retains EXTRA_COLS."""
    print("\n--- Running Footprint Reclamation Pipeline (Enriched) ---")
    group_a = df[df['junction_name'] != 'No Junction'].copy()
    group_b = df[df['junction_name'] == 'No Junction'].copy()

    centroids = group_a.groupby('junction_name')[['x_meters', 'y_meters']].mean().reset_index()
    centroids.rename(columns={'x_meters': 'cx', 'y_meters': 'cy'}, inplace=True)
    group_a   = group_a.merge(centroids, on='junction_name')
    group_a['dist'] = np.sqrt((group_a['x_meters'] - group_a['cx'])**2 + (group_a['y_meters'] - group_a['cy'])**2)
    radii     = group_a.groupby('junction_name')['dist'].quantile(0.95).reset_index(name='radius_95')
    j_lookup  = centroids.merge(radii, on='junction_name')

    dist_matrix   = cdist(group_b[['x_meters', 'y_meters']].values, j_lookup[['cx', 'cy']].values)
    valid_dists   = np.where(dist_matrix <= j_lookup['radius_95'].values, dist_matrix, np.inf)
    min_dist      = np.min(valid_dists, axis=1)
    best_junc_idx = np.argmin(valid_dists, axis=1)
    reclaimed_mask = min_dist != np.inf
    print(f" -> Reclaimed {reclaimed_mask.sum()} mislabeled points from Group B!")

    group_b['new_junction'] = 'No Junction'
    group_b.loc[reclaimed_mask, 'new_junction'] = j_lookup['junction_name'].iloc[best_junc_idx[reclaimed_mask]].values
    reclaimed_df = group_b[group_b['new_junction'] != 'No Junction'].copy()
    reclaimed_df['junction_name'] = reclaimed_df['new_junction']
    true_group_b  = group_b[group_b['new_junction'] == 'No Junction'].copy()
    final_group_a = pd.concat([group_a, reclaimed_df], ignore_index=True)

    group_a_summaries = []
    for junction_name, group in tqdm(final_group_a.groupby('junction_name'), desc="Aggregating Enriched Group A"):
        group_a_summaries.append({
            'hotspot_id': f"A-{len(group_a_summaries)+1}", 'label_name': junction_name,
            'label_type': 'confirmed_junction', 'total_violations': len(group),
            'centroid_lat': round(group['latitude'].mean(), 6),
            'centroid_long': round(group['longitude'].mean(), 6),
            'top_violation': _get_top_value(group['primary_violation']),
            'top_vehicle':   _get_top_value(group['vehicle_type'])
        })

    keep_a = ['latitude', 'longitude', 'junction_name'] + [c for c in EXTRA_COLS if c in final_group_a.columns]
    viz_a  = final_group_a[keep_a].copy()
    viz_a.rename(columns={'junction_name': 'Hotspot_Name'}, inplace=True)

    true_group_b_clustered, b_mapping, group_b_summaries = _run_street_grouped_hdbscan_enriched(true_group_b)

    keep_b = ['latitude', 'longitude', 'final_cluster_label'] + [c for c in EXTRA_COLS if c in true_group_b_clustered.columns]
    viz_b  = true_group_b_clustered[keep_b].copy()
    viz_b['Hotspot_Name'] = viz_b['final_cluster_label'].map(b_mapping)
    viz_b.drop(columns=['final_cluster_label'], inplace=True)

    summary_df  = pd.concat([pd.DataFrame(group_a_summaries), pd.DataFrame(group_b_summaries)], ignore_index=True)
    summary_df  = summary_df.sort_values(by='total_violations', ascending=False).reset_index(drop=True)
    granular_df = pd.concat([viz_a, viz_b], ignore_index=True)

    return summary_df, granular_df


# =============================================================
# TASK 1 — Full violation/vehicle distribution breakdowns
# =============================================================

def compute_distribution(series):
    """Returns {category: {count, pct}} dict for a categorical series."""
    counts = series.value_counts()
    total  = counts.sum()
    return {k: {'count': int(v), 'pct': round(v / total * 100, 2)} for k, v in counts.items()}

def enrich_summary_with_breakdowns(summary_df, granular_df):
    violation_breakdown = {}
    vehicle_breakdown   = {}
    for hotspot_name, grp in granular_df.groupby('Hotspot_Name'):
        violation_breakdown[hotspot_name] = compute_distribution(grp['primary_violation'].dropna())
        vehicle_breakdown[hotspot_name]   = compute_distribution(grp['vehicle_type'].dropna())
    summary_df = summary_df.copy()
    summary_df['violation_breakdown'] = summary_df['label_name'].map(violation_breakdown)
    summary_df['vehicle_breakdown']   = summary_df['label_name'].map(vehicle_breakdown)
    return summary_df


# =============================================================
# TASK 2 — Congestion-relevance flag, severity weights, score
# =============================================================

VIOLATION_SEVERITY = {
    # Congestion-related
    'WRONG PARKING':                                    (True, 5.0),
    'NO PARKING':                                       (True, 5.5),
    'PARKING IN A MAIN ROAD':                           (True, 9.5),
    'PARKING ON FOOTPATH':                              (True, 5.0),
    'DOUBLE PARKING':                                   (True, 9.0),
    'PARKING NEAR ROAD CROSSING':                       (True, 7.5),
    'PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS':        (True, 7.5),
    'PARKING OPPOSITE TO ANOTHER PARKED VEHICLE':       (True, 6.0),
    'PARKING OTHER THAN BUS STOP':                      (True, 7.0),
    'PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC':          (True, 7.0),
    'H T V PROHIBITED':                                 (True, 6.5),
    'AGAINST ONE WAY/NO ENTRY':                         (True, 8.0),
    'VIOLATING LANE DISIPLINE':                         (True, 7.0),
    'JUMPING TRAFFIC SIGNAL':                           (True, 7.5),
    'STOPING ON WHITE/STOP LINE':                       (True, 6.5),
    'OBSTRUCTING DRIVER':                               (True, 6.0),
    # Not congestion-related
    'DEFECTIVE NUMBER PLATE':                           (False, 0),
    'REFUSE TO GO FOR HIRE':                            (False, 0),
    'USING BLACK FILM/OTHER MATERIALS':                 (False, 0),
    'DEMANDING EXCESS FARE':                            (False, 0),
    'WITHOUT SIDE MIRROR':                              (False, 0),
    'FAIL TO USE SAFETY BELTS':                         (False, 0),
    'RIDER NOT WEARING HELMET':                         (False, 0),
    '2W/3W - USING MOBILE PHONE':                       (False, 0),
    'OTHER - USING MOBILE PHONE':                       (False, 0),
    'CARRYING LENGHTY MATERIAL':                        (False, 0),
    'U TURN PROHIBITED':                                (False, 0),
}

def compute_congestion_score(violation_breakdown):
    """Computes congestion_weighted_score from a violation_breakdown dict."""
    if not isinstance(violation_breakdown, dict):
        return 0.0
    score = 0.0
    for vtype, stats in violation_breakdown.items():
        entry = VIOLATION_SEVERITY.get(vtype)
        if entry and entry[0]:  # is_congestion_related
            score += stats['count'] * entry[1]
    return round(score, 2)

def enrich_summary_with_congestion_score(summary_df):
    summary_df = summary_df.copy()
    summary_df['congestion_weighted_score'] = summary_df['violation_breakdown'].apply(compute_congestion_score)
    return summary_df


# =============================================================
# TASK 3 — Peak-hour concentration per hotspot
# =============================================================

def compute_peak_hour_pct(granular_df):
    """Returns Series indexed by Hotspot_Name with peak_hour_pct values."""
    def _peak(grp):
        h = grp['hour']
        peak_mask = ((h >= 0) & (h < 7)) | ((h >= 19) & (h <= 23))
        return round(peak_mask.sum() / len(grp) * 100, 2) if len(grp) > 0 else 0.0
    return granular_df.groupby('Hotspot_Name').apply(_peak)

def enrich_summary_with_peak_hours(summary_df, granular_df):
    summary_df = summary_df.copy()
    peak_pct   = compute_peak_hour_pct(granular_df).rename('peak_hour_pct')
    summary_df = summary_df.merge(peak_pct.reset_index().rename(columns={'Hotspot_Name': 'label_name'}),
                                  on='label_name', how='left')
    summary_df['peak_hour_pct'] = summary_df['peak_hour_pct'].fillna(0.0)
    return summary_df


# =============================================================
# TASK 4 — Vehicle footprint weight per hotspot
# =============================================================

VEHICLE_FOOTPRINT_WEIGHT = {
    'HGV':                  10,
    'LORRY/GOODS VEHICLE':  9,
    'TANKER':               9,
    'MINI LORRY':           9,
    'TRACTOR':              9,
    'BUS (BMTC/KSRTC)':    8,
    'PRIVATE BUS':          8,
    'TOURIST BUS':          8,
    'SCHOOL VEHICLE':       7,
    'FACTORY BUS':          7,
    'LGV':                  7,
    'VAN':                  6,
    'TEMPO':                6,
    'JEEP':                 5,
    'MAXI-CAB':             5,
    'GOODS AUTO':           5,
    'CAR':                  4,
    'PASSENGER AUTO':       3,
    'SCOOTER':              2,
    'MOTOR CYCLE':          2,
    'MOPED':                1,
    'OTHERS':               4,
}

def compute_avg_vehicle_footprint(vehicle_breakdown):
    """Computes weighted-average footprint weight from a vehicle_breakdown dict."""
    if not isinstance(vehicle_breakdown, dict):
        return 0.0
    weighted_sum = 0.0
    total_count  = 0
    for vtype, stats in vehicle_breakdown.items():
        weight        = VEHICLE_FOOTPRINT_WEIGHT.get(vtype, 4)
        weighted_sum += stats['count'] * weight
        total_count  += stats['count']
    return round(weighted_sum / total_count, 4) if total_count > 0 else 0.0

def enrich_summary_with_vehicle_footprint(summary_df):
    summary_df = summary_df.copy()
    summary_df['avg_vehicle_footprint'] = summary_df['vehicle_breakdown'].apply(compute_avg_vehicle_footprint)
    return summary_df


# =============================================================
# TASK 5 — Impact Score (with confidence discount)
# =============================================================

IMPACT_WEIGHTS = {
    'total_violations':          0.30,
    'congestion_weighted_score': 0.30,
    'peak_hour_pct':             0.20,
    'avg_vehicle_footprint':     0.20,
}

CONFIDENCE_MIN_VOLUME = 100  # hotspot needs this many violations for full confidence

def compute_impact_score(summary_df, weights=None):
    if weights is None:
        weights = IMPACT_WEIGHTS
    summary_df = summary_df.copy()

    # Min-max normalize each component
    for col in weights:
        col_min = summary_df[col].min()
        col_max = summary_df[col].max()
        denom   = (col_max - col_min) if (col_max - col_min) > 0 else 1
        summary_df[f'_norm_{col}'] = (summary_df[col] - col_min) / denom

    # Raw weighted score (0-100)
    raw_score = sum(
        summary_df[f'_norm_{col}'] * w for col, w in weights.items()
    ) * 100

    # Confidence factor: min(1, total_violations / CONFIDENCE_MIN_VOLUME)
    # A hotspot with 1 violation gets factor 0.01; with 50 gets 0.50; with 100+ gets 1.0
    confidence = (summary_df['total_violations'] / CONFIDENCE_MIN_VOLUME).clip(upper=1.0)

    summary_df['raw_impact_score'] = raw_score.round(2)
    summary_df['confidence_factor'] = confidence.round(4)
    summary_df['impact_score']      = (raw_score * confidence).round(2)

    # Drop temp norm cols
    summary_df.drop(columns=[f'_norm_{col}' for col in weights], inplace=True)
    summary_df = summary_df.sort_values('impact_score', ascending=False).reset_index(drop=True)

    print("\nTop 20 Hotspots by Impact Score:")
    print("-" * 115)
    display_cols = ['label_name', 'label_type', 'total_violations', 'confidence_factor',
                    'congestion_weighted_score', 'peak_hour_pct',
                    'avg_vehicle_footprint', 'impact_score']
    print(summary_df[display_cols].head(20).to_string(index=False))
    print("-" * 115)
    return summary_df


# =============================================================
# TASK 6 — Repeat offender analysis
# =============================================================

def compute_repeat_offenders(granular_df):
    df = granular_df.copy()
    df['created_datetime']    = pd.to_datetime(df['created_datetime'],    errors='coerce')
    df['validation_timestamp'] = pd.to_datetime(df['validation_timestamp'], errors='coerce')

    records = []
    for vehicle_number, grp in df.groupby('vehicle_number'):
        if len(grp) <= 1:
            continue
        grp_sorted = grp.sort_values('created_datetime')

        # Avg days between consecutive occurrences
        times   = grp_sorted['created_datetime'].dropna()
        if len(times) > 1:
            gaps = times.diff().dropna().dt.total_seconds() / 86400
            avg_gap_days = round(gaps.mean(), 2)
        else:
            avg_gap_days = None

        # Avg validation turnaround hours
        valid_rows = grp[grp['validation_timestamp'].notna() & grp['created_datetime'].notna()].copy()
        valid_rows['turnaround_h'] = (
            valid_rows['validation_timestamp'] - valid_rows['created_datetime']
        ).dt.total_seconds() / 3600
        valid_rows = valid_rows[valid_rows['turnaround_h'] >= 0]
        avg_turnaround = round(valid_rows['turnaround_h'].mean(), 2) if len(valid_rows) > 0 else None

        records.append({
            'vehicle_number':                vehicle_number,
            'total_occurrences':             len(grp),
            'unique_creators':               grp['created_by_id'].nunique(),
            'first_seen':                    grp['created_datetime'].min(),
            'last_seen':                     grp['created_datetime'].max(),
            'avg_days_between_occurrences':  avg_gap_days,
            'avg_validation_turnaround_hours': avg_turnaround,
        })

    repeat_offenders_df = pd.DataFrame(records).sort_values('total_occurrences', ascending=False).reset_index(drop=True)
    print(f"\nRepeat Offenders — Top 15 (of {len(repeat_offenders_df)} total):")
    print("-" * 90)
    print(repeat_offenders_df.head(15).to_string(index=False))
    print("-" * 90)
    return repeat_offenders_df


# =============================================================
# TASK 7 — Police station / center code rollup
# =============================================================

def compute_police_station_rollup(summary_df, granular_df):
    # Merge impact_score onto granular_df via Hotspot_Name / label_name
    score_lookup = summary_df[['label_name', 'impact_score']].rename(columns={'label_name': 'Hotspot_Name'})
    merged       = granular_df.merge(score_lookup, on='Hotspot_Name', how='left')

    records = []
    for ps, grp in merged.groupby('police_station'):
        top_hotspot = (
            grp.groupby('Hotspot_Name')['impact_score'].mean().idxmax()
            if grp['impact_score'].notna().any() else None
        )
        records.append({
            'police_station':   ps,
            'total_violations': len(grp),
            'num_hotspots':     grp['Hotspot_Name'].nunique(),
            'avg_impact_score': round(grp['impact_score'].mean(), 2),
            'top_hotspot_name': top_hotspot,
        })

    police_station_rollup_df = (
        pd.DataFrame(records)
        .sort_values('avg_impact_score', ascending=False)
        .reset_index(drop=True)
    )
    print(f"\nPolice Station Rollup ({len(police_station_rollup_df)} stations):")
    print("-" * 110)
    print(police_station_rollup_df.to_string(index=False))
    print("-" * 110)
    return police_station_rollup_df


# =============================================================
# MASTER RUNNER — call this after pipelines complete
# =============================================================

def run_full_analysis(summary_df, granular_df):
    """Runs Tasks 1-7 in sequence, returns enriched summary_df and standalone outputs."""
    print("\n========== FULL ANALYSIS PIPELINE ==========")

    print("\n[Task 1] Computing violation/vehicle breakdowns...")
    summary_df = enrich_summary_with_breakdowns(summary_df, granular_df)

    print("[Task 2] Computing congestion-weighted scores...")
    summary_df = enrich_summary_with_congestion_score(summary_df)

    print("[Task 3] Computing peak-hour concentration...")
    summary_df = enrich_summary_with_peak_hours(summary_df, granular_df)

    print("[Task 4] Computing avg vehicle footprint weights...")
    summary_df = enrich_summary_with_vehicle_footprint(summary_df)

    print("[Task 5] Computing impact scores...")
    summary_df = compute_impact_score(summary_df)

    print("\n[Task 6] Computing repeat offenders...")
    repeat_offenders_df = compute_repeat_offenders(granular_df)

    print("\n[Task 7] Computing police station rollup...")
    police_station_rollup_df = compute_police_station_rollup(summary_df, granular_df)

    print("\n========== ANALYSIS COMPLETE ==========")
    return summary_df, repeat_offenders_df, police_station_rollup_df

In [7]:
file_path = "jan to may police violation_anonymized791b166.csv"
base_traffic_df = prepare_traffic_data(file_path)

# --- Pipeline A (enriched) ---
summary_standard, granular_standard = run_standard_hybrid_pipeline_enriched(base_traffic_df)
save_granular_map_html(granular_standard, "Bengaluru Hotspots (Standard Hybrid)", "approach_A")
save_summary_map_html(summary_standard, "Bengaluru Hotspot Summary (Standard Hybrid)", "approach_A_summary")
summary_standard, repeat_offenders_standard, ps_rollup_standard = run_full_analysis(summary_standard, granular_standard)

# # --- Pipeline B (enriched) ---
# summary_footprint, granular_footprint = run_footprint_pipeline_enriched(base_traffic_df)
# save_granular_map_html(granular_footprint, "Bengaluru Hotspots (Footprint Reclamation)", "approach_B")
# save_summary_map_html(summary_footprint, "Bengaluru Hotspot Summary (Footprint Reclamation)", "approach_B_summary")
# summary_footprint, repeat_offenders_footprint, ps_rollup_footprint = run_full_analysis(summary_footprint, granular_footprint)

Loading and preparing data...

--- Running Standard Hybrid Pipeline (Enriched) ---


Aggregating Known Junctions: 100%|██████████| 168/168 [00:00<00:00, 1296.64it/s]


 -> Batched 1976 small street groups.
 -> Batched 62 mega street groups (500+ pts).
 -> Running HDBSCAN on 147 large street groups...


Street-grouped HDBSCAN:   3%|▎         | 4/147 [00:00<00:03, 35.81it/s]

   Processing: '100 Feet Road' | points: 238
   Processing: '10th Cross Road' | points: 152
   Processing: '10th Main Road' | points: 173
   Processing: '11th Cross Road' | points: 374
   Processing: '11th Main Road' | points: 140
   Processing: '12th Cross Road' | points: 134
   Processing: '13th Cross Road' | points: 190
   Processing: '13th Main Road' | points: 255
   Processing: '14th Cross Road' | points: 111
   Processing: '16th Cross Road' | points: 160


...

 -> Successfully saved interactive map to: approach_A.html
Generating summary map for approach_A_summary...
 -> Successfully saved summary map to: approach_A_summary.html

========== FULL ANALYSIS PIPELINE ==========

[Task 1] Computing violation/vehicle breakdowns...


/tmp/ipykernel_365462/3898131404.py:423: DeprecationWarning:

*scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



[Task 2] Computing congestion-weighted scores...
[Task 3] Computing peak-hour concentration...


/tmp/ipykernel_365462/1954332688.py:324: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



[Task 4] Computing avg vehicle footprint weights...
[Task 5] Computing impact scores...

Top 20 Hotspots by Impact Score:
-------------------------------------------------------------------------------------------------------------------
                                                         label_name         label_type  total_violations  confidence_factor  congestion_weighted_score  peak_hour_pct  avg_vehicle_footprint  impact_score
                                     BTP051 - Safina Plaza Junction confirmed_junction             15449                1.0                    81207.0           7.31                 2.8385         65.55
                                        BTP082 - KR Market Junction confirmed_junction             11538                1.0                    58871.0          26.11                 2.9095         53.62
                                            BTP040 - Elite Junction confirmed_junction             10718                1.0                    56492.5   